# Construct pydantic model from text input

In [2]:
from pydantic_ai import Agent
agent = Agent(model="google-gla:gemini-2.5-flash")

result = await agent.run("Give me an IT employee working in sweden, shortly")
result

AgentRunResult(output="**Erik Karlsson**\n\n**Age:** 34\n**Location:** Stockholm, Sweden\n**Role:** DevOps Engineer\n**Company:** A fast-growing SaaS company specializing in e-commerce solutions.\n\n**Profile:** Erik is known for his methodical approach and calm demeanor. He focuses on automating software deployment pipelines, managing cloud infrastructure (primarily AWS), and ensuring system reliability and scalability. He's a strong advocate for good work-life balance, always enjoys his daily 'fika' with colleagues, and is fluent in both Swedish and English.")

In [35]:
from pydantic import BaseModel, Field


class Employee(BaseModel):
    name: str = Field(description="first and last name of the person")
    age: int
    salary: int = Field(gt=30_000, lt=50_000) # Description ger bra info till agenten som den kan använda 
    position: str
    role: str = Field(description="always include seniority level in the role (junior, senior, expert)")


result = await agent.run(
    "Give me an IT employee working in sweden", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Bjorn Borg', age=45, salary=45000, position='Software Engineer', role='Senior Software Engineer'))

In [36]:
result.output

EmployeeModel(name='Bjorn Borg', age=45, salary=45000, position='Software Engineer', role='Senior Software Engineer')

In [37]:
print(result.output)

name='Bjorn Borg' age=45 salary=45000 position='Software Engineer' role='Senior Software Engineer'


In [38]:
result.output.name, result.output.age, result.output.salary

('Bjorn Borg', 45, 45000)

In [39]:
result.output.model_dump_json()

'{"name":"Bjorn Borg","age":45,"salary":45000,"position":"Software Engineer","role":"Senior Software Engineer"}'

In [40]:
result = await agent.run(
    "Give me ten employees in AI and data engineering fields, so the roles can vary, salary must be between 30000 and 50000",
    output_type=list[Employee],
)
result

AgentRunResult(output=[Employee(name='Alice Wonderland', age=28, salary=45000, position='AI Engineer', role='Junior AI Engineer'), Employee(name='Bob Thebuilder', age=35, salary=49999, position='Data Engineer', role='Senior Data Engineer'), Employee(name='Charlie Chaplin', age=30, salary=40000, position='Machine Learning Engineer', role='Mid-level Machine Learning Engineer'), Employee(name='Diana Prince', age=40, salary=48000, position='Data Scientist', role='Expert Data Scientist'), Employee(name='Eve Harrington', age=25, salary=35000, position='AI Developer', role='Junior AI Developer'), Employee(name='Frank Sinatra', age=38, salary=47000, position='Big Data Engineer', role='Senior Big Data Engineer'), Employee(name='Grace Kelly', age=32, salary=39000, position='NLP Engineer', role='Mid-level NLP Engineer'), Employee(name='Harry Potter', age=45, salary=49000, position='AI Architect', role='Expert AI Architect'), Employee(name='Ivy League', age=29, salary=37000, position='ETL Develope

In [41]:
result.output

[Employee(name='Alice Wonderland', age=28, salary=45000, position='AI Engineer', role='Junior AI Engineer'),
 Employee(name='Bob Thebuilder', age=35, salary=49999, position='Data Engineer', role='Senior Data Engineer'),
 Employee(name='Charlie Chaplin', age=30, salary=40000, position='Machine Learning Engineer', role='Mid-level Machine Learning Engineer'),
 Employee(name='Diana Prince', age=40, salary=48000, position='Data Scientist', role='Expert Data Scientist'),
 Employee(name='Eve Harrington', age=25, salary=35000, position='AI Developer', role='Junior AI Developer'),
 Employee(name='Frank Sinatra', age=38, salary=47000, position='Big Data Engineer', role='Senior Big Data Engineer'),
 Employee(name='Grace Kelly', age=32, salary=39000, position='NLP Engineer', role='Mid-level NLP Engineer'),
 Employee(name='Harry Potter', age=45, salary=49000, position='AI Architect', role='Expert AI Architect'),
 Employee(name='Ivy League', age=29, salary=37000, position='ETL Developer', role='Juni

In [42]:
for employee in result.output:
    print(f"{employee.name} is {employee.age} years, earns {employee.salary} and have role {employee.role}")

Alice Wonderland is 28 years, earns 45000 and have role Junior AI Engineer
Bob Thebuilder is 35 years, earns 49999 and have role Senior Data Engineer
Charlie Chaplin is 30 years, earns 40000 and have role Mid-level Machine Learning Engineer
Diana Prince is 40 years, earns 48000 and have role Expert Data Scientist
Eve Harrington is 25 years, earns 35000 and have role Junior AI Developer
Frank Sinatra is 38 years, earns 47000 and have role Senior Big Data Engineer
Grace Kelly is 32 years, earns 39000 and have role Mid-level NLP Engineer
Harry Potter is 45 years, earns 49000 and have role Expert AI Architect
Ivy League is 29 years, earns 37000 and have role Junior ETL Developer
Jack Sparrow is 33 years, earns 42000 and have role Mid-level Database Engineer


## CV model - a more complex and nested model

In [9]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int


class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int


class CvModel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    educations: list[EducationModel]


result = await agent.run(
    "Create a fake person that is applying for a data engineering job",
    output_type=CvModel,
)
result


AgentRunResult(output=CvModel(name='John Doe', age=30, experiences=[ExperienceModel(title='Data Engineer', company='Tech Solutions', description='Developed and maintained ETL pipelines, managed data warehouses, and worked with big data technologies.', start_year=2020, end_year=2024), ExperienceModel(title='Junior Data Analyst', company='Data Insights Inc.', description='Assisted in data collection and cleaning, generated reports, and created dashboards.', start_year=2018, end_year=2020)], educations=[EducationModel(title='Master of Science in Data Science', education_area='Data Science', school='University of Technology', description='Specialized in big data processing and machine learning.', start_year=2019, end_year=2021), EducationModel(title='Bachelor of Science in Computer Science', education_area='Computer Science', school='State University', description='Focused on algorithms, data structures, and software development.', start_year=2014, end_year=2018)]))

In [10]:
result.output.name

'John Doe'

In [11]:
result.output.age

30

In [12]:
result.output.experiences[1].title, result.output.experiences[1].start_year

('Junior Data Analyst', 2018)

## (optional) Postprocessing - load into duckdb and unnesting

This part is optional, but a way to unnest the data and store it could be to use dlt to load the data into duckdb, followed by joining and unnesting.

Other approach could be to store into nosql such as mongodb.

In [13]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="cv_json_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging",
)

# Modeldump = Gör det till dict, 
info = pipeline.run(
    data=[result.output.model_dump()], loader_file_format="jsonl", table_name="cv_entries"
)

print(info)


Pipeline cv_json_duckdb load step completed in 0.15 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\eriku\Desktop\DataenginerSTI2024-2026\Github\AIgeneering_four_week_course\07_pydanticai\Kokchun\cv.duckdb location to store data
Load package 1764232396.3006203 is LOADED and contains no failed jobs


In [14]:
import duckdb 

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc;").df()
    cv_entries = conn.sql("FROM staging.cv_entries").df()
    educations = conn.sql("FROM staging.cv_entries__educations").df()
    experiences = conn.sql("FROM staging.cv_entries__experiences").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__educations,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experiences,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [15]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Alice Johnson,30,1764177512.8152063,SL08oihOkixoyQ
1,John Doe,30,1764232396.3006203,KEqRh28lOdykMw


In [16]:
educations

,title,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Master of Science in Data Science,Data Science,State University of Technology,"Specialized in big data technologies, machine ...",2017,2018,SL08oihOkixoyQ,0,j5Y5j8zfhI0iVg
1,Bachelor of Science in Computer Science,Computer Science,City College of Engineering,"Focused on algorithms, data structures, databa...",2014,2017,SL08oihOkixoyQ,1,+CFvwxGAxUCjuw
2,Master of Science in Data Science,Data Science,University of Technology,Specialized in big data processing and machine...,2019,2021,KEqRh28lOdykMw,0,wtcW1zF/UgFY7A
3,Bachelor of Science in Computer Science,Computer Science,State University,"Focused on algorithms, data structures, and so...",2014,2018,KEqRh28lOdykMw,1,b9jYZ9bIxNHzLQ


In [17]:
experiences

,title,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Senior Data Engineer,TechInnovate Solutions,Led the design and implementation of scalable ...,2021,2024,SL08oihOkixoyQ,0,RwT8M70QBW07gQ
1,Data Engineer,DataStream Analytics,Developed and maintained data ingestion system...,2018,2021,SL08oihOkixoyQ,1,3ZHM/mbuThrojg
2,Data Engineer,Tech Solutions,"Developed and maintained ETL pipelines, manage...",2020,2024,KEqRh28lOdykMw,0,AfY+vTjSqsQmPg
3,Junior Data Analyst,Data Insights Inc.,"Assisted in data collection and cleaning, gene...",2018,2020,KEqRh28lOdykMw,1,raFCSWhj1cF4kA


In [18]:
duckdb.sql("""
    SELECT 
        cv.name, 
        cv.age, 
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN educations e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id
    

""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,Alice Johnson,30,DataStream Analytics,Developed and maintained data ingestion system...,2018,2021,Master of Science in Data Science,Data Science,State University of Technology,2017,2018
1,Alice Johnson,30,DataStream Analytics,Developed and maintained data ingestion system...,2018,2021,Bachelor of Science in Computer Science,Computer Science,City College of Engineering,2014,2017
2,John Doe,30,Data Insights Inc.,"Assisted in data collection and cleaning, gene...",2018,2020,Master of Science in Data Science,Data Science,University of Technology,2019,2021
3,John Doe,30,Data Insights Inc.,"Assisted in data collection and cleaning, gene...",2018,2020,Bachelor of Science in Computer Science,Computer Science,State University,2014,2018
4,Alice Johnson,30,TechInnovate Solutions,Led the design and implementation of scalable ...,2021,2024,Master of Science in Data Science,Data Science,State University of Technology,2017,2018
5,Alice Johnson,30,TechInnovate Solutions,Led the design and implementation of scalable ...,2021,2024,Bachelor of Science in Computer Science,Computer Science,City College of Engineering,2014,2017
6,John Doe,30,Tech Solutions,"Developed and maintained ETL pipelines, manage...",2020,2024,Master of Science in Data Science,Data Science,University of Technology,2019,2021
7,John Doe,30,Tech Solutions,"Developed and maintained ETL pipelines, manage...",2020,2024,Bachelor of Science in Computer Science,Computer Science,State University,2014,2018


##Adding another model

In [43]:
from typing import Literal

class Department(BaseModel):
    name: Literal["IT", "HR", "Marketing"] = Field(description="name of the department")
    description: str = Field(description="short description of the department")
    employees: list[Employee]

class Company(BaseModel):
    name: str = Field(description="name of the company")
    departments: list[Department]
    description: str = Field(description="describe company in two sentences")

result = await agent.run(
    ("Give 2-3 employees in each department",
     "Company should be called CoolCompany and sell ice creams"
     ),
    output_type=Company
    
)

In [52]:
company = result.output
company.__dict__.keys()

dict_keys(['name', 'departments', 'description'])

In [56]:
print(company)

name='CoolCompany' departments=[Department(name='IT', description='Handles all technology infrastructure and software development.', employees=[Employee(name='John Doe', age=35, salary=45000, position='Software Engineer', role='Senior Software Engineer'), Employee(name='Jane Smith', age=28, salary=40000, position='IT Support Specialist', role='Junior IT Support Specialist'), Employee(name='Mike Brown', age=42, salary=48000, position='IT Manager', role='Expert IT Manager')]), Department(name='HR', description='Manages employee relations, recruitment, and company culture.', employees=[Employee(name='Sarah Connor', age=38, salary=43000, position='HR Business Partner', role='Senior HR Business Partner'), Employee(name='David Lee', age=30, salary=42000, position='Recruiter', role='Junior Recruiter'), Employee(name='Emily White', age=45, salary=47000, position='HR Director', role='Expert HR Director')]), Department(name='Marketing', description="Promotes the company's products and manages br

In [45]:
company.name

'CoolCompany'

In [46]:
[department.name for department in company.departments]

['IT', 'HR', 'Marketing']

In [47]:

departments = [department for department in company.departments]
departments[0].employees

[Employee(name='John Doe', age=35, salary=45000, position='Software Engineer', role='Senior Software Engineer'),
 Employee(name='Jane Smith', age=28, salary=40000, position='IT Support Specialist', role='Junior IT Support Specialist'),
 Employee(name='Mike Brown', age=42, salary=48000, position='IT Manager', role='Expert IT Manager')]

In [48]:
company.model_dump()

{'name': 'CoolCompany',
 'departments': [{'name': 'IT',
   'description': 'Handles all technology infrastructure and software development.',
   'employees': [{'name': 'John Doe',
     'age': 35,
     'salary': 45000,
     'position': 'Software Engineer',
     'role': 'Senior Software Engineer'},
    {'name': 'Jane Smith',
     'age': 28,
     'salary': 40000,
     'position': 'IT Support Specialist',
     'role': 'Junior IT Support Specialist'},
    {'name': 'Mike Brown',
     'age': 42,
     'salary': 48000,
     'position': 'IT Manager',
     'role': 'Expert IT Manager'}]},
  {'name': 'HR',
   'description': 'Manages employee relations, recruitment, and company culture.',
   'employees': [{'name': 'Sarah Connor',
     'age': 38,
     'salary': 43000,
     'position': 'HR Business Partner',
     'role': 'Senior HR Business Partner'},
    {'name': 'David Lee',
     'age': 30,
     'salary': 42000,
     'position': 'Recruiter',
     'role': 'Junior Recruiter'},
    {'name': 'Emily White